In [1]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf

In [2]:
df = pd.read_csv(r"modded_metadata.csv")

In [3]:
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,data\images\ISIC_0027419.jpg,2
1,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,data\images\ISIC_0026769.jpg,2
2,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,data\images\ISIC_0031633.jpg,2
3,HAM_0002761,ISIC_0029176,bkl,histo,60.0,male,face,data\images\ISIC_0029176.jpg,2
4,HAM_0005132,ISIC_0025837,bkl,histo,70.0,female,back,data\images\ISIC_0025837.jpg,2


In [4]:
df.shape

(7470, 9)

In [5]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['dx'])
class_names = le.classes_
print(dict(zip(class_names, range(len(class_names)))))

{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}


In [6]:
image_dir = r'data\images'
df['path'] = df['image_id'].apply(lambda x: os.path.join(image_dir, f"{x}.jpg"))

assert all(os.path.exists(p) for p in df['path'].head()), "Missing files!"

## Stratified Train/Test/Split   
stratfied to maintain class distribution because of the class imbalance   
verify the distribution with `normalize=True` to see percentages – they should be similar.

In [7]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print(f"Train samples: {len(train_df)}")
print(f"Val samples: {len(val_df)}")
print(train_df['dx'].value_counts(normalize=True))
print(val_df['dx'].value_counts(normalize=True))

Train samples: 5976
Val samples: 1494
dx
nv       0.723394
bkl      0.097390
mel      0.082162
bcc      0.043842
akiec    0.030455
vasc     0.013052
df       0.009705
Name: proportion, dtype: float64
dx
nv       0.722892
bkl      0.097055
mel      0.082329
bcc      0.043507
akiec    0.030790
vasc     0.013387
df       0.010040
Name: proportion, dtype: float64


In [8]:
assert set(train_df['lesion_id']).isdisjoint(set(val_df['lesion_id'])), "Leakage detected!"

# Data Pipeline

In [9]:
# gloabal parameters
IMG_SIZE = 224          # ResNet/EfficientNet typically use 224x224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE  # Let TensorFlow decide parallelism

Most pre-trained models we’ll use expect images of 224×224 pixels.

In [10]:
def load_image(path, label):
    """Load an image from a file path and preprocess it."""
    # 1. Read the file as binary
    image = tf.io.read_file(path)
    # 2. Decode JPEG (3 color channels: RGB)
    image = tf.image.decode_jpeg(image, channels=3)
    # 3. Resize to a fixed size (bilinear interpolation by default)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    # 4. Normalize pixel values from [0,255] to [0,1]
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [11]:
def augment(image, label):
    """Apply random transformations to the training image."""
    # Randomly flip horizontally
    image = tf.image.random_flip_left_right(image)
    # Randomly flip vertically
    image = tf.image.random_flip_up_down(image)
    # Random rotation: 0, 90, 180, or 270 degrees (optional, but useful)
    # We can use tf.image.rot90 with a random k
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k=k)
    # Random brightness and contrast adjustments (within small ranges)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    # Keep pixel values in [0,1]
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

-   **Flips**: Lesions can be photographed from any orientation.
-   **Rotation**: Same reasoning – there’s no canonical “up” for a skin lesion.
-   **Brightness/contrast**: Lighting varies across clinics and devices.
-   `clip_by_value` prevents values from leaving the \[0,1\] range after brightness/contrast changes.`

### Build the tf.data.Dataset for training and validation

In [12]:
def create_dataset(dataframe, training=False):
    """Create a tf.data Dataset from a DataFrame with 'path' and 'label' columns."""
    paths = dataframe['path'].values
    labels = dataframe['label'].values

    # Create a dataset of (path, label) tuples
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    # Load and preprocess images
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if training:
        # Apply augmentation after loading
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        # Shuffle the training data to avoid learning order biases
        ds = ds.shuffle(buffer_size=len(dataframe))

    # Batch the data
    ds = ds.batch(BATCH_SIZE)
    # Prefetch to overlap data preparation and model execution
    ds = ds.prefetch(AUTOTUNE)

    return ds

# Create the actual datasets
train_ds = create_dataset(train_df, training=True)
val_ds   = create_dataset(val_df, training=False)

-   `from_tensor_slices` takes the two arrays and creates a dataset that yields one `(path, label)` at a time.
-   `map(load_image)` applies our load function to every element. `num_parallel_calls=AUTOTUNE` lets TensorFlow decide how many threads to use, maximising speed.
-   Training only: `map(augment)` adds the random transformations.
-   `shuffle(buffer_size)` randomises the order of the training samples. The buffer should be large enough (here we use the whole dataset size) so that the model sees a different order each epoch.
-   `batch` groups samples into mini‑batches of 32.
-   `prefetch(AUTOTUNE)` makes the next batch ready while the model is training on the current one, reducing GPU idle time.